# Selective next-range-bar direction research

This notebook consumes the reusable `range_bars_ml` library; feature construction and training logic do not live here.


In [1]:
# Reload local modelling modules so re-running this cell picks up source edits.
import importlib
import range_bars_ml as _range_bars_package
import range_bars_ml.model as _range_bars_model
import range_bars_ml.pipeline as _range_bars_pipeline

importlib.reload(_range_bars_model)
importlib.reload(_range_bars_pipeline)
importlib.reload(_range_bars_package)

from range_bars_ml import (
    ExperimentConfig,
    FeatureConfig,
    ModelConfig,
    load_range_bars,
    prepare_dataset,
    run_experiment,
)
from range_bars_ml.model import signal_metrics
import numpy as np
import polars as pl


In [2]:
# Data selection
DATA_PATH = r"C:/Code/Trading/2026/ml/data/renko_bars_100.parquet"

# Causal feature definition
FEATURE_CONFIG = FeatureConfig(
    windows=range(1,20,1),
    time_column="close_time",
)

# LightGBM and probability-calibration definition
MODEL_CONFIG = ModelConfig(
    calibration_fraction=0.10,
    random_state=18616,
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=100,
)

# Require a probability edge above the development-period up rate.
# Set to 0.0 to disable this additional anti-trend gate.
EDGE_MARGIN = 0.02

# Chronological evaluation and selective-signal definition
EXPERIMENT_CONFIG = ExperimentConfig(
    feature=FEATURE_CONFIG,
    model=MODEL_CONFIG,
    test_fraction=0.20,
    n_folds=5,
    # Zero permits an all-abstain strategy if no calls have positive net score.
    minimum_coverage=0.0,
    abstention_penalty=-0.001,
    edge_margin=EDGE_MARGIN,
)

# Inspection and artifact choices
FEATURE_PREVIEW_ROWS = 10
FEATURE_EXPORT_PATH = None  # e.g. r"C:\Code\Trading\2026\ml\data\range_bar_features.parquet"
FEATURE_EXPORT_COMPRESSION = "zstd"
MODEL_ARTIFACT_PATH = "range_bar_model.joblib"


In [3]:
all_bars = load_range_bars(DATA_PATH)
bars = all_bars.filter(pl.col("close_time") < pl.datetime(2025, 1, 1, time_zone='UTC'))
#bars = bars.slice(offset=DATA_SLICE_OFFSET, length=DATA_SLICE_LENGTH)


In [4]:
bars.shape

(57481, 12)

In [5]:
bars[-1]

first_trade_id,last_trade_id,close_time,open,high,low,close,total_sold_volume,total_bought_volume,total_sold_dollar_volume,total_bought_dollar_volume,timestamp
i64,i64,"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, UTC]"
4358983732,4359194927,2024-12-31 17:37:44.475 UTC,94712.55,95047.71,93765.19,93765.19,590.83172,468.36727,5.5847e7,4.4311e7,2024-12-31 16:32:02.214 UTC


In [6]:
# Inspect the fully causal, next-bar-labelled training frame.
feature_df, feature_names = prepare_dataset(bars, config=FEATURE_CONFIG)
feature_df.head(FEATURE_PREVIEW_ROWS)

# Set FEATURE_EXPORT_PATH above to persist the complete feature frame.
if FEATURE_EXPORT_PATH:
    feature_df.write_parquet(FEATURE_EXPORT_PATH, compression=FEATURE_EXPORT_COMPRESSION)


In [7]:
# Metrics include score (+1 correct, -1 wrong, -0.001 abstain).
result = run_experiment(bars, config=EXPERIMENT_CONFIG)

# The edge gate is applied inside run_experiment before its single final-test score.
dataset, _ = prepare_dataset(bars, config=FEATURE_CONFIG)
test = dataset[result.split_plan.test.tolist()]
development_up_rate = result.model.metadata["development_up_rate"]


In [8]:
# Compare against simple regime/trend baselines on the same final test.
test_target = test["target"].to_numpy()
baseline_metrics = {
    "always_up": signal_metrics(np.ones(len(test_target)), test_target, 0.5, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_down": signal_metrics(np.zeros(len(test_target)), test_target, 1.01, 0.5, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
    "always_abstain": signal_metrics(np.full(len(test_target), 0.5), test_target, 1.01, -0.01, abstention_penalty=EXPERIMENT_CONFIG.abstention_penalty),
}
score_comparison = pl.DataFrame([
    {"strategy": "model_edge_gated", **result.test_metrics},
    *({"strategy": name, **metrics} for name, metrics in baseline_metrics.items()),
]).select(["strategy", "score", "mean_score", "signals", "correct", "wrong", "abstentions", "precision"])

print(f"Development up rate: {development_up_rate:.3%}; edge gates: long >= {result.model.long_threshold:.3f}, short <= {result.model.short_threshold:.3f}")
score_comparison

Development up rate: 51.044%; edge gates: long >= 0.530, short <= 0.490


strategy,score,mean_score,signals,correct,wrong,abstentions,precision
str,f64,f64,f64,f64,f64,f64,f64
"""model_edge_gated""",439.0,0.038197,11493.0,5966.0,5527.0,0.0,0.519099
"""always_up""",207.0,0.018011,11493.0,5850.0,5643.0,0.0,0.509005
"""always_down""",-207.0,-0.018011,11493.0,5643.0,5850.0,0.0,0.490995
"""always_abstain""",-11.493,-0.001,0.0,0.0,0.0,11493.0,NaN


In [9]:
result.model.save(MODEL_ARTIFACT_PATH)


In [10]:
from range_bars_ml.model import SelectiveSignalModel
model = SelectiveSignalModel.load(MODEL_ARTIFACT_PATH)
feature_config = FeatureConfig(**model.metadata["feature_config"])

In [11]:
new_bars =all_bars.filter(pl.col("close_time") >= pl.datetime(2025, 1, 1, time_zone='UTC'))
dataset, _ = prepare_dataset(new_bars, feature_config)

In [12]:
predictions = model.predict(dataset)
signals = dataset.select("target").hstack(predictions)

In [13]:
metrics = signal_metrics(
      signals["probability_up"].to_numpy(),
      signals["target"].to_numpy(),
      model.long_threshold,
      model.short_threshold,
      abstention_penalty=model.metadata.get("abstention_penalty", -0.001),
  )

In [14]:
print(metrics)

{'coverage': 0.3483398987056837, 'signals': 1238.0, 'correct': 681.0, 'wrong': 557.0, 'abstentions': 2316.0, 'score': 121.684, 'mean_score': 0.03423860438942037, 'precision': 0.5500807754442649, 'long_coverage': 0.17726505346088914, 'long_precision': 0.5619047619047619, 'long_signals': 630.0, 'short_coverage': 0.1710748452447946, 'short_precision': 0.537828947368421, 'short_signals': 608.0}


In [15]:
bars[-1]


first_trade_id,last_trade_id,close_time,open,high,low,close,total_sold_volume,total_bought_volume,total_sold_dollar_volume,total_bought_dollar_volume,timestamp
i64,i64,"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, UTC]"
4358983732,4359194927,2024-12-31 17:37:44.475 UTC,94712.55,95047.71,93765.19,93765.19,590.83172,468.36727,5.5847e7,4.4311e7,2024-12-31 16:32:02.214 UTC


In [16]:
import pandas as pd
from range_bars_ml import SelectiveSignalModel

model = SelectiveSignalModel.load(MODEL_ARTIFACT_PATH)

importance = (
  pd.DataFrame({
      "feature": model.feature_columns,
      "split_count": model.estimator.feature_importances_,
      "gain": model.estimator.booster_.feature_importance(importance_type="gain"),
  })
  .sort_values("gain", ascending=False)
)

print(importance.head(30).to_string(index=False))

                                 feature  split_count         gain
                              bar_return          268 20042.335898
                            log_return_1          122  2180.420970
                            log_return_3          144  1558.796227
                            log_return_2          121  1397.677682
                           bar_range_pct          133  1378.480342
                            return_std_2          124  1369.300607
                            log_return_8           96  1019.388772
                            return_std_3           94  1017.099545
                            log_return_4           92   980.657670
                           log_return_10           77   943.215920
                            log_return_9           95   936.807783
         bar_time_share_window_19_lag_18           86   905.735872
                            log_return_6           82   896.050701
                            return_std_4           82   894.91